# Simulación de Impacto Económico del Error de Predicción

**Fase 4 — Análisis de valor de negocio**

Simula el costo económico que enfrenta una PYME agroindustrial cuando usa las predicciones
de los modelos GE, GM_v3 y XGBoost para planificar su aprovisionamiento de limón.

- **Merma:** el modelo predijo más de lo real → la PYME compró de más → producto perecible perdido
- **Quiebre de stock:** el modelo predijo menos de lo real → la PYME compró de menos → venta perdida

Foco principal: **enero 2025**, el shock máximo del periodo de test (1021% variación).

In [ ]:
import os, platform, sys
import numpy as np
import pandas as pd
import joblib
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.facecolor'] = 'white'

# ── Parámetros de la simulación PYME típica ──
PRECIO_LIMON_KG    = 2.50      # S/ por kg en chacra (precio shock enero 2025)
CAPACIDAD_PYME     = 50_000    # kg por mes que puede procesar la PYME
COSTO_MERMA_KG     = 1.80      # S/ por kg de merma (producto perecible no procesado)
COSTO_STOCKOUT_KG  = 3.20      # S/ por kg de quiebre de stock (venta perdida)

UMBRAL_SHOCK_PCT   = 20        # % variación mensual para considerar shock

print(f'Parámetros PYME:')
print(f'  Precio limón chacra : S/ {PRECIO_LIMON_KG:.2f}/kg')
print(f'  Capacidad mensual   : {CAPACIDAD_PYME:,} kg')
print(f'  Costo merma         : S/ {COSTO_MERMA_KG:.2f}/kg')
print(f'  Costo stockout      : S/ {COSTO_STOCKOUT_KG:.2f}/kg')

In [ ]:
def detect_project_root() -> Path:
    if platform.system() == 'Windows':
        p = Path('C:/Machine-learming/Machine-Learning-Multimodal--Agro-NLP-Clima-')
        if p.exists():
            return p
    for candidate in [
        Path('/mnt/c/Machine-learming/Machine-Learning-Multimodal--Agro-NLP-Clima-'),
        Path.home() / 'Machine-learming' / 'Machine-Learning-Multimodal--Agro-NLP-Clima-',
    ]:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('No se encontró la raíz del proyecto.')

ROOT = detect_project_root()

# Archivos de predicciones (modelos originales, n_train~44)
GE_PRED_PATH   = ROOT / 'resultados' / 'ge'      / 'ge_predicciones.csv'
GM_PRED_PATH   = ROOT / 'resultados' / 'gm_v3'   / 'gm_v3_predicciones.csv'
XGB_PRED_PATH  = ROOT / 'resultados' / 'xgboost' / 'xgb_predicciones.csv'

# Scaler de fase 2 original (corresponde a los modelos n_train=44)
SCALER_PATH    = ROOT / 'notebooks' / 'fase2' / 'scalers' / 'standard_scaler_fase2.joblib'

# Master dataset para detectar shocks
MASTER_PATH    = ROOT / 'data' / 'processed' / 'master_dataset_fase2_multivariado.csv'

for label, p in [('GE pred', GE_PRED_PATH), ('GM_v3 pred', GM_PRED_PATH),
                 ('XGB pred', XGB_PRED_PATH), ('Scaler', SCALER_PATH),
                 ('Master', MASTER_PATH)]:
    print(f'{label:12s}: {"OK" if p.exists() else "FALTA"} — {p.name}')

## 1. Scaler de Fase 2 original

Los valores en los CSVs de predicciones son z-scores (media provincial) de `produccion_t`.
Para convertir a toneladas reales: `tons = z × scale + mean`.

In [ ]:
scaler_fase2 = joblib.load(SCALER_PATH)

# produccion_t es la primera feature (index 0)
MEAN_PROD  = scaler_fase2.mean_[0]
SCALE_PROD = scaler_fase2.scale_[0]

print(f'Scaler Fase 2 para produccion_t:')
print(f'  mean  = {MEAN_PROD:.4f} t')
print(f'  scale = {SCALE_PROD:.4f} t')
print(f'  features: {scaler_fase2.feature_names_in_[0]}  (confirmado)')

def desn_tons(z):
    """Z-score fase 2 -> toneladas."""
    return z * SCALE_PROD + MEAN_PROD

def desn_kg(z):
    """Z-score fase 2 -> kilogramos."""
    return desn_tons(z) * 1000

## 2. Carga de predicciones

In [ ]:
# GE: columnas fecha, real, prediccion, error_abs
df_ge = pd.read_csv(GE_PRED_PATH, parse_dates=['fecha'])
df_ge = df_ge.rename(columns={'prediccion': 'pred'})
df_ge['modelo'] = 'GE'

# GM_v3: columnas fecha, real, pred_gm_v3
df_gm = pd.read_csv(GM_PRED_PATH, parse_dates=['fecha'])
df_gm = df_gm.rename(columns={'pred_gm_v3': 'pred'})
df_gm['modelo'] = 'GM_v3'

# XGBoost: columnas fecha, real, pred_xgb
df_xgb = pd.read_csv(XGB_PRED_PATH, parse_dates=['fecha'])
df_xgb = df_xgb.rename(columns={'pred_xgb': 'pred'})
df_xgb['modelo'] = 'XGBoost'

for label, df in [('GE', df_ge), ('GM_v3', df_gm), ('XGBoost', df_xgb)]:
    print(f'{label:8s}: {len(df)} meses  [{df.fecha.min().date()} → {df.fecha.max().date()}]')

## 3. Detección de meses de shock

Los shocks se detectan sobre la serie de z-scores (media provincial), donde las variaciones
porcentuales son amplificadas cuando los valores cruzan cerca de cero. Esto es coherente con
la metodología original que identificó 8 meses de shock en el test, incluyendo enero 2025
con 1021% de variación.

La desnormalización a toneladas se usa solo para la simulación económica.

In [ ]:
df_master = pd.read_csv(MASTER_PATH, parse_dates=['fecha_evento'])
nacional = df_master.groupby('fecha_evento')['produccion_t'].mean().sort_index()

# Shocks sobre z-scores (metodología original: pct_change > 20% en z-scores)
nacional_var_zscore = nacional.pct_change().abs() * 100

# Periodo de test: desde la primera fecha disponible en las predicciones
fecha_min_test = min(df_ge.fecha.min(), df_xgb.fecha.min())
test_mask = nacional.index >= fecha_min_test

shock_mask = test_mask & (nacional_var_zscore > UMBRAL_SHOCK_PCT)
fechas_shock = set(nacional.index[shock_mask])

# Desnormalizar para mostrar contexto
nacional_tons = desn_tons(nacional)

print(f'Serie nacional: {len(nacional)} meses')
print(f'Periodo test  : desde {fecha_min_test.date()}')
print(f'Shocks en test: {len(fechas_shock)} meses con variación >{UMBRAL_SHOCK_PCT}%')
print()

print(f'{"Fecha":<12} {"z-score":>10} {"Prod (t)":>10} {"Var z (%)":>10} {"Shock":>6}')
print('-' * 52)
for fecha in sorted(nacional.index[test_mask]):
    z = nacional[fecha]
    t = nacional_tons[fecha]
    v = nacional_var_zscore[fecha]
    s = '● SI' if fecha in fechas_shock else ''
    print(f'{str(fecha.date()):<12} {z:>10.4f} {t:>10.2f} {v:>9.1f}% {s:>6}')

## 4. Simulación económica: desnormalización y cálculo de pérdidas

Para cada modelo y mes:
1. Desnormalizar predicción y valor real a toneladas
2. Convertir a kg
3. Error absoluto en kg
4. Si predijo MÁS → merma (S/ 1.80/kg) | Si predijo MENOS → stockout (S/ 3.20/kg)
5. Pérdida en soles

In [ ]:
def calcular_impacto(df_pred):
    """Calcula el impacto económico para un DataFrame de predicciones."""
    rows = []
    for _, r in df_pred.iterrows():
        real_t = desn_tons(r['real'])
        pred_t = desn_tons(r['pred'])
        real_kg = real_t * 1000
        pred_kg = pred_t * 1000
        error_kg = abs(pred_kg - real_kg)

        if pred_kg > real_kg:
            tipo = 'Merma'
            perdida = error_kg * COSTO_MERMA_KG
        else:
            tipo = 'Stockout'
            perdida = error_kg * COSTO_STOCKOUT_KG

        es_shock = r['fecha'] in fechas_shock

        rows.append({
            'fecha': r['fecha'],
            'modelo': r['modelo'],
            'real_t': real_t,
            'pred_t': pred_t,
            'error_t': abs(real_t - pred_t),
            'real_kg': real_kg,
            'pred_kg': pred_kg,
            'error_kg': error_kg,
            'tipo_error': tipo,
            'perdida_soles': perdida,
            'es_shock': es_shock,
        })
    return pd.DataFrame(rows)

# Calcular impacto para los 3 modelos
impacto_ge  = calcular_impacto(df_ge[['fecha', 'real', 'pred', 'modelo']])
impacto_gm  = calcular_impacto(df_gm[['fecha', 'real', 'pred', 'modelo']])
impacto_xgb = calcular_impacto(df_xgb[['fecha', 'real', 'pred', 'modelo']])

impacto_all = pd.concat([impacto_ge, impacto_gm, impacto_xgb], ignore_index=True)

print(f'Registros totales: {len(impacto_all)}')
print(f'  GE     : {len(impacto_ge)} meses')
print(f'  GM_v3  : {len(impacto_gm)} meses')
print(f'  XGBoost: {len(impacto_xgb)} meses')

## 5. Enero 2025 — Shock máximo (1021% variación)

Tabla comparativa detallada para el mes de mayor volatilidad del periodo de test.

In [ ]:
ene25 = impacto_all[impacto_all.fecha == '2025-01-01'].copy()

if len(ene25) == 0:
    print('Ningún modelo tiene predicción para enero 2025')
else:
    print('=' * 95)
    print('  IMPACTO ECONÓMICO — ENERO 2025 (shock máximo: 1021% variación)')
    print('=' * 95)
    print(f'  {"Modelo":<10} {"Pred (t)":>10} {"Real (t)":>10} {"Error (t)":>10} '
          f'{"Error (kg)":>11} {"Tipo":>10} {"Pérdida (S/)":>14}')
    print('  ' + '-' * 91)

    for _, r in ene25.iterrows():
        print(f'  {r.modelo:<10} {r.pred_t:>10.2f} {r.real_t:>10.2f} {r.error_t:>10.2f} '
              f'{r.error_kg:>11,.0f} {r.tipo_error:>10} {r.perdida_soles:>14,.2f}')

    print('=' * 95)

    # Contexto
    mejor = ene25.loc[ene25.perdida_soles.idxmin()]
    peor  = ene25.loc[ene25.perdida_soles.idxmax()]
    print(f'\n  Mejor modelo en enero 2025 : {mejor.modelo} (S/ {mejor.perdida_soles:,.2f})')
    print(f'  Peor modelo en enero 2025  : {peor.modelo} (S/ {peor.perdida_soles:,.2f})')
    print(f'  Diferencia                 : S/ {peor.perdida_soles - mejor.perdida_soles:,.2f}')
    print(f'\n  Nota: GM_v3 no tiene predicción para enero 2025 (su test inicia en marzo 2025).')

## 6. Todos los meses de shock del periodo de test

Tabla completa con pérdida económica para cada mes con variación >20%.

In [ ]:
shock_data = impacto_all[impacto_all.es_shock].sort_values(['fecha', 'modelo']).copy()

print('=' * 105)
print(f'  IMPACTO ECONÓMICO EN MESES DE SHOCK (variación >{UMBRAL_SHOCK_PCT}%)')
print('=' * 105)
print(f'  {"Fecha":<12} {"Modelo":<10} {"Pred (t)":>10} {"Real (t)":>10} {"Error (t)":>10} '
      f'{"Error (kg)":>11} {"Tipo":>10} {"Pérdida (S/)":>14}')
print('  ' + '-' * 101)

prev_fecha = None
for _, r in shock_data.iterrows():
    if prev_fecha is not None and r.fecha != prev_fecha:
        print('  ' + '-' * 101)
    prev_fecha = r.fecha
    print(f'  {str(r.fecha.date()):<12} {r.modelo:<10} {r.pred_t:>10.2f} {r.real_t:>10.2f} '
          f'{r.error_t:>10.2f} {r.error_kg:>11,.0f} {r.tipo_error:>10} '
          f'{r.perdida_soles:>14,.2f}')

print('=' * 105)

## 7. Pérdida acumulada en meses de shock por modelo

In [ ]:
resumen = (
    shock_data
    .groupby('modelo')
    .agg(
        n_shocks=('fecha', 'count'),
        error_total_kg=('error_kg', 'sum'),
        perdida_total=('perdida_soles', 'sum'),
        n_merma=('tipo_error', lambda x: (x == 'Merma').sum()),
        n_stockout=('tipo_error', lambda x: (x == 'Stockout').sum()),
        perdida_promedio=('perdida_soles', 'mean'),
        perdida_max=('perdida_soles', 'max'),
    )
    .sort_values('perdida_total')
)

print('=' * 100)
print('  PÉRDIDA ACUMULADA EN MESES DE SHOCK POR MODELO')
print('=' * 100)
print(f'  {"Modelo":<10} {"Shocks":>7} {"Merma":>6} {"S.Out":>6} '
      f'{"Error tot (kg)":>15} {"Pérdida total":>15} {"Pérdida prom":>14} '
      f'{"Pérdida max":>14}')
print('  ' + '-' * 96)

for modelo, r in resumen.iterrows():
    print(f'  {modelo:<10} {r.n_shocks:>7} {r.n_merma:>6} {r.n_stockout:>6} '
          f'{r.error_total_kg:>15,.0f} {r.perdida_total:>13,.2f} S/ '
          f'{r.perdida_promedio:>12,.2f} S/ {r.perdida_max:>12,.2f} S/')

print('=' * 100)

# Pérdida total (todos los meses, no solo shocks)
resumen_total = (
    impacto_all
    .groupby('modelo')
    .agg(
        n_meses=('fecha', 'count'),
        perdida_total=('perdida_soles', 'sum'),
    )
    .sort_values('perdida_total')
)

print(f'\n  PÉRDIDA TOTAL (todos los meses del test):')
print('  ' + '-' * 50)
for modelo, r in resumen_total.iterrows():
    pct_shock = 0
    if modelo in resumen.index:
        pct_shock = resumen.loc[modelo, 'perdida_total'] / r.perdida_total * 100
    print(f'  {modelo:<10} {r.n_meses:>3} meses  S/ {r.perdida_total:>12,.2f}  '
          f'({pct_shock:.0f}% en shocks)')

## 8. Visualización

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors_modelo = {'GE': '#e74c3c', 'GM_v3': '#2ecc71', 'XGBoost': '#3498db'}

# 1. Pérdida en enero 2025
ax = axes[0]
ene_data = impacto_all[impacto_all.fecha == '2025-01-01'].set_index('modelo')
if len(ene_data) > 0:
    modelos_ene = ene_data.index.tolist()
    vals = ene_data['perdida_soles'].values
    bars = ax.bar(modelos_ene, vals,
                  color=[colors_modelo.get(m, 'gray') for m in modelos_ene],
                  alpha=0.85, edgecolor='white', linewidth=0.8)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.02,
                f'S/ {v:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Pérdida Enero 2025\n(shock máximo)', fontsize=11, fontweight='bold')
ax.set_ylabel('Pérdida (S/)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}'))

# 2. Pérdida acumulada en shocks
ax = axes[1]
modelos_ord = resumen.index.tolist()
vals = resumen['perdida_total'].values
bars = ax.bar(modelos_ord, vals,
              color=[colors_modelo.get(m, 'gray') for m in modelos_ord],
              alpha=0.85, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, v + max(vals)*0.02,
            f'S/ {v:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_title('Pérdida acumulada\n(meses de shock)', fontsize=11, fontweight='bold')
ax.set_ylabel('Pérdida (S/)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}'))

# 3. Tipo de error (merma vs stockout) por modelo
ax = axes[2]
tipo_resumen = (
    shock_data
    .groupby(['modelo', 'tipo_error'])['perdida_soles']
    .sum()
    .unstack(fill_value=0)
)
tipo_resumen.plot(kind='bar', ax=ax, color=['#e67e22', '#8e44ad'],
                  alpha=0.85, edgecolor='white', linewidth=0.8)
ax.set_title('Merma vs Stockout\n(meses de shock)', fontsize=11, fontweight='bold')
ax.set_ylabel('Pérdida (S/)')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=0)
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}'))

plt.suptitle('Simulación de Impacto Económico — PYME Agroindustrial (limón)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'resultados' / 'simulacion_impacto_economico.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Guardado -> resultados/simulacion_impacto_economico.png')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: Producción real vs predicha (en toneladas)
ax = axes[0]
for modelo, df_m in impacto_all.groupby('modelo'):
    df_m = df_m.sort_values('fecha')
    ax.plot(df_m.fecha, df_m.pred_t, label=f'{modelo} (pred)',
            color=colors_modelo[modelo], ls='--', marker='s', ms=4, lw=1.5)

# Real (tomar de GE que tiene más meses)
real_series = impacto_ge.sort_values('fecha')
ax.plot(real_series.fecha, real_series.real_t, label='Real',
        color='black', lw=2, marker='o', ms=5)

# Marcar shocks
for fecha in sorted(fechas_shock):
    if fecha >= real_series.fecha.min():
        ax.axvline(fecha, color='red', alpha=0.15, lw=8)

ax.set_title('Producción desnormalizada (t) — Real vs Predicciones', fontsize=11)
ax.set_ylabel('Producción (toneladas)')
ax.legend(fontsize=8, ncol=2)

# Panel 2: Pérdida económica por mes
ax = axes[1]
width = pd.Timedelta(days=6)
offsets = {'GE': -width, 'GM_v3': pd.Timedelta(0), 'XGBoost': width}

for modelo, df_m in impacto_all.groupby('modelo'):
    df_m = df_m.sort_values('fecha')
    ax.bar(df_m.fecha + offsets[modelo], df_m.perdida_soles,
           width=width, label=modelo, color=colors_modelo[modelo],
           alpha=0.8, edgecolor='white', linewidth=0.5)

for fecha in sorted(fechas_shock):
    if fecha >= real_series.fecha.min():
        ax.axvline(fecha, color='red', alpha=0.15, lw=8)

ax.set_title('Pérdida económica por mes (S/) — franjas rojas = meses de shock', fontsize=11)
ax.set_ylabel('Pérdida (S/)')
ax.set_xlabel('Fecha')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'S/ {x:,.0f}'))

plt.tight_layout()
plt.savefig(ROOT / 'resultados' / 'simulacion_timeline.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Guardado -> resultados/simulacion_timeline.png')

## 9. Tabla completa — Todos los meses, todos los modelos

In [ ]:
tabla = impacto_all.sort_values(['fecha', 'modelo']).copy()
tabla['fecha_str'] = tabla['fecha'].dt.strftime('%Y-%m')

print('=' * 115)
print('  TABLA COMPLETA — IMPACTO ECONÓMICO POR MODELO Y MES')
print('=' * 115)
print(f'  {"Fecha":<9} {"Modelo":<10} {"Pred (t)":>10} {"Real (t)":>10} '
      f'{"Error (t)":>10} {"Error (kg)":>11} {"Tipo":>10} '
      f'{"Pérdida (S/)":>14} {"Shock":>6}')
print('  ' + '-' * 111)

prev_fecha = None
for _, r in tabla.iterrows():
    if prev_fecha is not None and r.fecha_str != prev_fecha:
        print('  ' + '-' * 111)
    prev_fecha = r.fecha_str
    shock_mark = '●' if r.es_shock else ''
    print(f'  {r.fecha_str:<9} {r.modelo:<10} {r.pred_t:>10.2f} {r.real_t:>10.2f} '
          f'{r.error_t:>10.2f} {r.error_kg:>11,.0f} {r.tipo_error:>10} '
          f'{r.perdida_soles:>14,.2f} {shock_mark:>6}')

print('=' * 115)

# Totales
print(f'\n  TOTALES POR MODELO:')
for modelo in ['GE', 'GM_v3', 'XGBoost']:
    dm = impacto_all[impacto_all.modelo == modelo]
    print(f'  {modelo:<10} {len(dm):>2} meses | '
          f'Error total: {dm.error_kg.sum():>10,.0f} kg | '
          f'Pérdida total: S/ {dm.perdida_soles.sum():>12,.2f}')

## 10. Conclusiones

### Hallazgos clave

| Aspecto | Resultado |
|---|---|
| **Enero 2025** | GE: S/ 4,444 vs XGBoost: S/ 4,938 de pérdida por merma (GM_v3 sin datos) |
| **Tipo dominante** | 100% merma — todos los modelos sobreestiman sistemáticamente la producción |
| **Pérdida total en shocks** | GM_v3: S/ 10,893 (3 shocks) < XGBoost: S/ 16,587 (6) < GE: S/ 20,801 (6) |
| **Pérdida total test** | GM_v3: S/ 19,496 (6m) < XGBoost: S/ 23,729 (10m) < GE: S/ 33,889 (10m) |
| **Implicación** | XGBoost tiene menor MAE en z-score, pero GE acumula más pérdida por sobreestimación persistente |

### Sesgo de sobreestimación

Los tres modelos predicen consistentemente **más** producción que la real durante todo
el periodo de test. Esto indica un sesgo sistemático: los modelos fueron entrenados en un
periodo de producción relativamente estable (2021–2024) y no capturan la tendencia
descendente de 2024-2025.

### Parámetros de la simulación

```
Precio limón chacra : S/ 2.50/kg
Capacidad PYME      : 50,000 kg/mes
Costo merma         : S/ 1.80/kg (producto perecible no procesado)
Costo stockout      : S/ 3.20/kg (venta perdida + penalidad)
Scaler              : Fase 2 original (mean=16.32 t, scale=27.99 t)
Shock detection     : variación >20% en z-scores (media provincial)
```